In [0]:
import pandas as pd

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
df = spark.table("workspace.silver.superstore").toPandas()


spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_customers")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_customers (
  customer_id   STRING,
  customer_name STRING,
  segment       STRING
)
""")

dim_customers = (
    df.reindex(columns=["customer_id", "customer_name", "segment"])
      .dropna(subset=["customer_id"])
      .drop_duplicates(subset=["customer_id"])
      .sort_values("customer_name", ignore_index=True)
)

spark.createDataFrame(dim_customers).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_customers")



In [0]:

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_products")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_products (
  product_id   STRING,
  product_name STRING,
  category     STRING,
  sub_category STRING
)
""")

dim_products = (
    df.reindex(columns=["product_id", "product_name", "category", "sub_category"])
      .dropna(subset=["product_id"])
      .drop_duplicates(subset=["product_id"])
      .sort_values("product_name", ignore_index=True)
)

spark.createDataFrame(dim_products).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_products")



In [0]:

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_location")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_location (
  postal_code INT,
  city        STRING,
  state       STRING,
  region      STRING,
  country     STRING
)
""")

dim_location = (
    df.reindex(columns=["postal_code", "city", "state", "region", "country"])
      .dropna(subset=["postal_code"])
      .drop_duplicates(subset=["postal_code"])
      .sort_values("state", ignore_index=True)
)

spark.createDataFrame(dim_location).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_location")



In [0]:

spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_time")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_time (
  date_key     BIGINT,
  date         TIMESTAMP,
  year         INT,
  quarter      INT,
  month        INT,
  month_name   STRING,
  day          INT,
  day_name     STRING,
  week_of_year BIGINT,
  is_weekend   BOOLEAN
)
""")

date_series = pd.to_datetime(df["order_date"], errors="coerce") \
                .dropna().drop_duplicates().sort_values()

dim_time = pd.DataFrame({"date": date_series})
dim_time["date_key"]     = dim_time["date"].dt.strftime("%Y%m%d").astype(int)
dim_time["year"]         = dim_time["date"].dt.year
dim_time["quarter"]      = dim_time["date"].dt.quarter
dim_time["month"]        = dim_time["date"].dt.month
dim_time["month_name"]   = dim_time["date"].dt.month_name()
dim_time["day"]          = dim_time["date"].dt.day
dim_time["day_name"]     = dim_time["date"].dt.day_name()
dim_time["week_of_year"] = dim_time["date"].dt.isocalendar().week.astype(int)
dim_time["is_weekend"]   = dim_time["day_name"].isin(["Saturday", "Sunday"])

dim_time = dim_time[["date_key", "date", "year", "quarter", "month",
                      "month_name", "day", "day_name", "week_of_year", "is_weekend"]]

spark.createDataFrame(dim_time).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_time")




In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_orders")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_orders (
  order_id             STRING,
  customer_id          STRING,
  product_id           STRING,
  postal_code          INT,
  date_key             BIGINT,
  ship_mode            STRING,
  retail_sales_people  STRING,
  returned             STRING,
  sales                DOUBLE,
  quantity             INT,
  discount             DOUBLE,
  profit               DOUBLE
)
""")

dates = pd.to_datetime(df["order_date"], errors="coerce")

fact_orders = (
    df.reindex(columns=[
        "order_id", "customer_id", "product_id", "postal_code",
        "ship_mode", "retail_sales_people", "returned",
        "sales", "quantity", "discount", "profit"
    ])
    .assign(date_key=dates.dt.strftime("%Y%m%d").astype("Int64"))
    .reindex(columns=[
        "order_id", "customer_id", "product_id", "postal_code", "date_key",
        "ship_mode", "retail_sales_people", "returned",
        "sales", "quantity", "discount", "profit"
    ])
    .dropna(subset=["order_id", "product_id"])
)

spark.createDataFrame(fact_orders).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_orders")



In [0]:
for table in ["dim_customers", "dim_products", "dim_location", "dim_time", "fact_orders"]:
    count = spark.table(f"workspace.gold.{table}").count()
    print(f"workspace.gold.{table}: {count} filas")